# 05 — External UCI HAR Sequential-Sensing Replay

External replay using the UCI Human Activity Recognition Using Smartphones dataset. The primary analysis is binary and theorem-aligned in structure; the six-class section is exploratory. This is not physical safety validation.

**Execution modes:** `SMOKE_TEST=1` performs a small offline code-path check; default settings run a moderate benchmark; `FULL_RUN=1` enables publication-scale Monte Carlo. Smoke outputs are verification-only and must not be reported as experimental results.

In [1]:
from pathlib import Path
import os, sys, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
if not (ROOT/'src'/'gated_benchmarks.py').exists():
    for cand in [HERE, *HERE.parents]:
        if (cand/'src'/'gated_benchmarks.py').exists():
            ROOT = cand; break
sys.path.insert(0, str(ROOT/'src'))
( ROOT/'results').mkdir(exist_ok=True)
( ROOT/'figures').mkdir(exist_ok=True)
FULL_RUN = os.getenv('FULL_RUN','0') == '1'
SMOKE_TEST = os.getenv('SMOKE_TEST','0') == '1'
print('ROOT =', ROOT)
print('FULL_RUN =', FULL_RUN, 'SMOKE_TEST =', SMOKE_TEST)


ROOT = /mnt/data/gated_certification_benchmarks
FULL_RUN = False SMOKE_TEST = True


In [2]:
import io, zipfile, urllib.request, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from gated_benchmarks import mean_ci, wilson_interval

UCI_URL='https://archive.ics.uci.edu/static/public/240/human%2Bactivity%2Brecognition%2Busing%2Bsmartphones.zip'


In [3]:
def make_mock_har(seed=0,n_train=1200,n_test=500,n_features=80,n_classes=6):
    # Smoke-only proxy: accelerometer classes overlap substantially; gyroscope adds separation.
    rng=np.random.default_rng(seed); half=n_features//2; names=[]
    for j in range(n_features):
        names.append((f'tBodyAcc-mean()-X{j}' if j<half else f'tBodyGyro-mean()-X{j}'))
    direction_acc=rng.normal(size=half); direction_acc/=np.linalg.norm(direction_acc)
    direction_gyro=rng.normal(size=n_features-half); direction_gyro/=np.linalg.norm(direction_gyro)
    coords=np.linspace(-1,1,n_classes)
    means=np.zeros((n_classes,n_features))
    for k,c in enumerate(coords):
        means[k,:half]=0.45*c*direction_acc
        means[k,half:]=1.8*c*direction_gyro
    # Make activities 2 and 3 especially close under weak sensing but distinguishable with gyro.
    means[1,:half]=-.80*direction_acc; means[2,:half]=.80*direction_acc
    means[1,half:]=-1.50*direction_gyro; means[2,half:]=1.50*direction_gyro
    def sample(n):
        y=rng.integers(1,n_classes+1,n); X=np.vstack([rng.normal(means[k-1],.85) for k in y]); return pd.DataFrame(X,columns=names),pd.Series(y)
    Xtr,ytr=sample(n_train); Xte,yte=sample(n_test); return Xtr,ytr,Xte,yte,names

def load_uci_har(use_real=True):
    if not use_real:
        return make_mock_har()
    cache=ROOT/'results'/'uci_har.zip'
    if not cache.exists():
        print('Downloading UCI HAR (~58 MB)...')
        urllib.request.urlretrieve(UCI_URL,cache)
    with zipfile.ZipFile(cache) as z:
        prefix='UCI HAR Dataset/'
        feat=pd.read_csv(z.open(prefix+'features.txt'),sep=r'\s+',header=None,names=['id','name'])['name'].astype(str).tolist()
        # Make duplicates unique without altering modality substrings.
        feat=[f'{name}__{i}' for i,name in enumerate(feat)]
        Xtr=pd.read_csv(z.open(prefix+'train/X_train.txt'),sep=r'\s+',header=None,names=feat)
        ytr=pd.read_csv(z.open(prefix+'train/y_train.txt'),header=None)[0]
        Xte=pd.read_csv(z.open(prefix+'test/X_test.txt'),sep=r'\s+',header=None,names=feat)
        yte=pd.read_csv(z.open(prefix+'test/y_test.txt'),header=None)[0]
    return Xtr,ytr,Xte,yte,feat

USE_REAL = not SMOKE_TEST
try:
    Xtr,ytr,Xte,yte,feature_names=load_uci_har(USE_REAL)
    DATA_SOURCE='UCI HAR real data' if USE_REAL else 'mock smoke-test data'
except Exception as e:
    warnings.warn(f'Real download failed ({e}); using mock data for code validation only.')
    Xtr,ytr,Xte,yte,feature_names=make_mock_har(); DATA_SOURCE='mock fallback — NOT REPORTABLE'
print(DATA_SOURCE, Xtr.shape, Xte.shape, sorted(ytr.unique()))


mock smoke-test data (1200, 80) (500, 80) [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


## 1. Define weak vs strong sensor channels
Weak channel uses accelerometer-derived features. Strong channel uses both accelerometer and gyroscope-derived features. The official UCI HAR dataset (Reyes-Ortiz et al., DOI: 10.24432/C54S4K) contains 10,299 windows from 30 volunteers performing six activities, using waist-mounted accelerometer and gyroscope sensing. The original subject-level train/test partition is retained.

In [4]:
weak_cols=[c for c in Xtr.columns if 'Acc' in c and 'Gyro' not in c]
strong_cols=[c for c in Xtr.columns if ('Acc' in c or 'Gyro' in c)]
if len(weak_cols)<5: weak_cols=list(Xtr.columns[:max(5,Xtr.shape[1]//2)])
if len(strong_cols)<=len(weak_cols): strong_cols=list(Xtr.columns)
print('weak features',len(weak_cols),'strong features',len(strong_cols))


weak features 40 strong features 80


In [5]:
def fit_density(X,y,cols,n_components):
    sc=StandardScaler().fit(X[cols]); Z=sc.transform(X[cols]); ncomp=min(n_components,Z.shape[1],max(2,Z.shape[0]-1)); pca=PCA(n_components=ncomp,random_state=0).fit(Z); Q=pca.transform(Z); g=GaussianNB(var_smoothing=1e-6).fit(Q,y); return sc,pca,g,cols

def class_loglik(model,Xrow):
    sc,pca,g,cols=model; q=pca.transform(sc.transform(pd.DataFrame([Xrow[cols].to_numpy()],columns=cols)))[0]
    var=g.var_; theta=g.theta_; return -0.5*np.sum(np.log(2*np.pi*var)+(q[None,:]-theta)**2/var,axis=1)

def batch_class_loglik(model,X):
    # Vectorized density table: rows x classes. Faster than transforming each replayed window repeatedly.
    sc,pca,g,cols=model
    Q=pca.transform(sc.transform(X[cols]))
    var=g.var_[None,:,:]; theta=g.theta_[None,:,:]
    QQ=Q[:,None,:]
    return -0.5*np.sum(np.log(2*np.pi*var)+(QQ-theta)**2/var,axis=2)


## 2. Binary external replay: WALKING_UPSTAIRS (2) vs WALKING_DOWNSTAIRS (3)
This deliberately harder pair uses the public train split only for calibration and the public test split only for replay. Because the action-conditioned densities are **estimated** rather than known, this is an external generalization check, not a proof of the theorem assumptions.

In [6]:
classes=[2,3]; tr=ytr.isin(classes); te=yte.isin(classes)
Xbtr=Xtr.loc[tr].reset_index(drop=True); ybtr=ytr.loc[tr].reset_index(drop=True)
Xbte=Xte.loc[te].reset_index(drop=True); ybte=yte.loc[te].reset_index(drop=True)
wm=fit_density(Xbtr,ybtr,weak_cols,10); sm=fit_density(Xbtr,ybtr,strong_cols,20)
LLW_B=batch_class_loglik(wm,Xbte); LLS_B=batch_class_loglik(sm,Xbte)
print('test counts',ybte.value_counts().to_dict())


test counts {2: 82, 3: 80}


In [7]:
def replay_binary(policy='gated',delta=.01,alpha=.05,n_trials=300,seed=0,max_steps=500):
    rng=np.random.default_rng(seed); A=math.log(1/delta); B=math.log(1/alpha); rows=[]
    cl=list(wm[2].classes_); i0=cl.index(2); i1=cl.index(3)
    pools={h:np.flatnonzero(ybte.to_numpy()==h) for h in classes}
    for h in classes:
        for tr in range(n_trials):
            S=M=0.; unlocked=(policy=='oracle'); nw=ns=0; dec=None
            for t in range(1,max_steps+1):
                idx=int(rng.choice(pools[h])); use_strong=(unlocked and policy!='weak_only') or policy=='oracle'
                ll=LLS_B[idx] if use_strong else LLW_B[idx]; inc=float(ll[i0]-ll[i1]); S+=inc; M=max(M,S)
                if use_strong: ns+=1
                else: nw+=1
                if policy=='gated' and (not unlocked) and M>=B: unlocked=True
                if S>=A: dec=2; break
                if S<=-A: dec=3; break
            if dec is None: dec=2 if S>=0 else 3
            rows.append({'true_class':h,'decision':dec,'error':int(dec!=h),'tau':nw+ns,'Nweak':nw,'Nstrong':ns,'unlocked':int(unlocked)})
    return pd.DataFrame(rows)

nrep=20 if SMOKE_TEST else (300 if not FULL_RUN else 2000)
rows=[]
for pol in ['gated','oracle','weak_only']:
    r=replay_binary(pol,n_trials=nrep,seed=10+['gated','oracle','weak_only'].index(pol))
    for h in classes:
        q=r[r.true_class==h]; rows.append({'policy':pol,'true_class':h,'mean_tau':q.tau.mean(),'error_rate':q.error.mean(),'mean_Nweak':q.Nweak.mean(),'mean_Nstrong':q.Nstrong.mean(),'unlock_rate':q.unlocked.mean()})
extbin=pd.DataFrame(rows); display(extbin); extbin.to_csv(ROOT/'results'/'uci_har_binary_replay.csv',index=False)


,policy,true_class,mean_tau,error_rate,mean_Nweak,mean_Nstrong,unlock_rate
0,gated,2,3.10,0.0,2.45,0.65,1.0
1,gated,3,5.85,0.0,5.45,0.40,0.2
2,oracle,2,1.05,0.0,0.00,1.05,1.0
3,oracle,3,1.35,0.0,0.00,1.35,1.0
4,weak_only,2,3.05,0.0,3.05,0.00,0.0
5,weak_only,3,4.40,0.0,4.40,0.00,0.0


## 3. Six-class exploratory replay
A second section keeps all six activities. It uses a confidence-gated switch from weak to strong sensor models. This is **not** the matched binary theorem; it is an external multi-hypothesis stress test.

In [8]:
wm6=fit_density(Xtr,ytr,weak_cols,3); sm6=fit_density(Xtr,ytr,strong_cols,8)
LLW6=batch_class_loglik(wm6,Xte); LLS6=batch_class_loglik(sm6,Xte)
classes6=list(wm6[2].classes_); pools6={h:np.flatnonzero(yte.to_numpy()==h) for h in classes6}

def replay_multi(delta=.1,cert_margin=2.0,n_trials_per_class=25,seed=0,max_steps=200):
    rng=np.random.default_rng(seed); A=math.log((len(classes6)-1)/delta); rows=[]
    for h in classes6:
        for tr in range(n_trials_per_class):
            scores=np.zeros(len(classes6)); unlocked=False; nw=ns=0; dec=None
            for t in range(1,max_steps+1):
                idx=int(rng.choice(pools6[h])); ll=LLS6[idx] if unlocked else LLW6[idx]; scores+=ll
                if unlocked: ns+=1
                else: nw+=1
                order=np.argsort(scores)[::-1]; margin=scores[order[0]]-scores[order[1]]
                if not unlocked and margin>=cert_margin: unlocked=True
                if margin>=A: dec=classes6[order[0]]; break
            if dec is None: dec=classes6[int(np.argmax(scores))]
            rows.append({'true_class':h,'decision':dec,'error':int(dec!=h),'tau':nw+ns,'Nweak':nw,'Nstrong':ns,'unlocked':int(unlocked)})
    return pd.DataFrame(rows)
rm=replay_multi(n_trials_per_class=(3 if SMOKE_TEST else (25 if not FULL_RUN else 100)),seed=55)
summary6=rm.groupby('true_class').agg(mean_tau=('tau','mean'),error_rate=('error','mean'),strong_use=('Nstrong','mean')).reset_index(); display(summary6); summary6.to_csv(ROOT/'results'/'uci_har_sixclass_replay.csv',index=False)


,true_class,mean_tau,error_rate,strong_use
0,1,57.666667,0.333333,42.333333
1,2,79.666667,0.666667,53.333333
2,3,107.666667,0.333333,18.333333
3,4,25.333333,0.000000,3.000000
4,5,99.333333,0.666667,12.666667
5,6,124.000000,0.666667,49.333333


## 4. PRISM follow-on protocol
For a robotics-specific replay, replace the UCI channel definitions with PRISM modalities: weak = robot state + low-risk force/torque summary; strong = tactile + richer force/torque/RGB-D features. Keep the wording **offline replay/counterfactual instantiation**, because a passive dataset cannot prove physical safety of an action gate. PRISM is large, so it is intentionally not downloaded automatically by this lightweight notebook.

## Reporting guardrail
For the paper, call this section **External sequential-sensing replay**. Do not call it 'real-world safety validation.' The public data establish that the gating machinery can be instantiated with estimated action/sensor-conditioned models; they do not establish the physical safety semantics assumed by a real robot certificate.